In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
def predict_on_new_data():
    print("Loading saved models...")
    save_dir = "E:\\Ai\\edu\\Smart Learning Tracker\\models\\saved_models"
    
    if not os.path.exists(save_dir) or not os.listdir(save_dir):
        print("Models not found. Please run train_ml_pipeline.py first.")
        return
        
    clf = joblib.load(os.path.join(save_dir, "lesson_level_classifier.pkl"))
    model_features = joblib.load(os.path.join(save_dir, "model_features.pkl"))
    
    # Load new/incoming data (we'll use preprocessed_data for simulation)
    print("Loading input data...")
    # Typically this would just be new event data
    data_path = "E:\\Ai\\edu\\Smart Learning Tracker\\data\\processed\\preprocessed_data.csv"
    df = pd.read_csv(data_path)
    
    features = ['watched', 'watch_time_min', 'attempts', 'time_spent']
    categorical_features = ['mistake_type']
    
    print("Pre-processing input features...")
    X = df[features + categorical_features].copy()
    X = pd.get_dummies(X, columns=categorical_features, drop_first=True)
    
    # Reindex columns to match training features exactly (in case a mistake type wasn't present in new data)
    for col in model_features:
        if col not in X.columns:
            X[col] = 0
            
    # Remove extra columns that weren't inside the training set
    X = X[model_features]
    
    print("Running Model Predictions...")
    # Predict the lesson level instead of using purely rule-based logic
    df['predicted_lesson_level'] = clf.predict(X)
    
    print("Determining Next Step Actions...")
    
    next_steps = []
    for idx, row in df.iterrows():
        level = row['predicted_lesson_level']
        
        if level == "weak":
            action = "watch_video + practice_easy"
        elif level == "medium":
            action = "practice_additional"
        else:
            action = "next_lecture / advanced_content"
            
        next_steps.append({
            "student_id": row['student_id'],
            "course_id": row['course_id'],
            "lecture_id": row['lecture_id'],
            "predicted_lesson_level": level,
            "next_action": action
        })
        
    df_report = pd.DataFrame(next_steps)
    
    output_path = "E:\\Ai\\edu\\Smart Learning Tracker\\data\\processed\\ml_cumulative_report.csv"
    df_report.to_csv(output_path, index=False)
    
    print(f"Prediction successful! Next step ML-driven report saved to: {output_path}")

if __name__ == "__main__":
    predict_on_new_data()


Loading saved models...
Loading input data...
Pre-processing input features...
Running Model Predictions...
Determining Next Step Actions...
Prediction successful! Next step ML-driven report saved to: E:\Ai\edu\Smart Learning Tracker\data\processed\ml_cumulative_report.csv
